## Step 1 — Define the routes
### FAQ → returns, refunds, shipping, payment, tracking, cancellation

### PRODUCT → price, brand, rating, discount, product recommendations

### SMALL TALK → hello, who are you, how are you

## Cell 1

In [1]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

D:\Gen AI\Project 3\ecommerce-chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2274.35it/s]


## Cell 2 — Route examples

In [3]:
route_examples = {
    "faq": [
        "What is your return policy?",
        "How can I track my order?",
        "What payment methods are accepted?",
        "How long does a refund take?",
        "Can I cancel my order?",
        "Do you offer international shipping?",
        "What should I do if my product is damaged?"
    ],

    "product": [
        "Show me Nike shoes under 5000",
        "What are the cheapest shoes available?",
        "Show products with rating above 4.5",
        "Which Campus shoes have the highest discount?",
        "Suggest Puma shoes under 3000",
        "Show me the top rated products"
    ],

    "smalltalk": [
        "Hello",
        "How are you?",
        "What is your name?",
        "Who are you?",
        "Are you a robot?",
        "What can you do?"
    ]
}

## Step 2 — Precompute embeddings

In [4]:
route_embeddings = {}

for route_name, examples in route_examples.items():
    route_embeddings[route_name] = model.encode(
        examples,
        convert_to_tensor=True
    )

for route_name, embeddings in route_embeddings.items():
    print(route_name, embeddings.shape)

faq torch.Size([7, 384])
product torch.Size([6, 384])
smalltalk torch.Size([6, 384])


## Step 3 — Route one question

In [7]:
query = "Can I get my money back if I return the shoes?"

query_embedding = model.encode(
    query,
    convert_to_tensor=True
)

for route_name, embeddings in route_embeddings.items():

    scores = util.cos_sim(
        query_embedding,
        embeddings
    )

    print(route_name, scores.max().item())

faq 0.46588772535324097
product 0.4362791180610657
smalltalk 0.12303775548934937


## Step 4 — Turn it into an actual router

In [8]:
def semantic_route(query, threshold=0.50):

    query_embedding = model.encode(
        query,
        convert_to_tensor=True
    )

    route_scores = {}

    for route_name, embeddings in route_embeddings.items():

        similarities = util.cos_sim(
            query_embedding,
            embeddings
        )

        route_scores[route_name] = similarities.max().item()

    best_route = max(
        route_scores,
        key=route_scores.get
    )

    best_score = route_scores[best_route]

    if best_score < threshold:
        return "unknown", best_score, route_scores

    return best_route, best_score, route_scores

In [10]:
queries = [
    "Can I return a damaged product?",
    "Show me Campus shoes below 1200",
    "Hey, how are you?",
    "Explain quantum physics"
]

for query in queries:
    route, score, all_scores = semantic_route(query)

    print("\nQuery:", query)
    print("Route:", route)
    print("Confidence:", round(score, 3))
    print("Scores:", {
        k: round(v, 3)
        for k, v in all_scores.items()
    })


Query: Can I return a damaged product?
Route: faq
Confidence: 0.794
Scores: {'faq': 0.794, 'product': 0.303, 'smalltalk': 0.206}

Query: Show me Campus shoes below 1200
Route: product
Confidence: 0.706
Scores: {'faq': 0.099, 'product': 0.706, 'smalltalk': 0.11}

Query: Hey, how are you?
Route: smalltalk
Confidence: 0.814
Scores: {'faq': 0.171, 'product': 0.122, 'smalltalk': 0.814}

Query: Explain quantum physics
Route: unknown
Confidence: 0.113
Scores: {'faq': 0.078, 'product': -0.002, 'smalltalk': 0.113}


## Cell 6 — Import semantic-router

In [11]:
from semantic_router import Route
from semantic_router.routers import SemanticRouter
from semantic_router.encoders import HuggingFaceEncoder

## Cell 7 — Create the encoder

In [12]:
encoder = HuggingFaceEncoder(
    name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2277.96it/s]


## Cell 8 — Define routes

In [13]:
faq_route = Route(
    name="faq",
    utterances=route_examples["faq"]
)

product_route = Route(
    name="product",
    utterances=route_examples["product"]
)

smalltalk_route = Route(
    name="smalltalk",
    utterances=route_examples["smalltalk"]
)

## Cell 9 — Build the router

In [14]:
router = SemanticRouter(
    encoder=encoder,
    routes=[
        faq_route,
        product_route,
        smalltalk_route
    ],
    auto_sync="local"
)

2026-09-30 23:22:26 WARNING semantic_router No index provided. Using default LocalIndex.
2026-09-30 23:22:26 WARNING semantic_router No config is written for LocalIndex.


## Cell 10 — Test it

In [15]:
test_queries = [
    "Can I return a damaged product?",
    "Show me Campus shoes below 1200",
    "Hey, how are you?"
]

for query in test_queries:
    result = router(query)

    print("Query:", query)
    print("Route:", result.name)
    print()

Query: Can I return a damaged product?
Route: None

Query: Show me Campus shoes below 1200
Route: product

Query: Hey, how are you?
Route: None



In [16]:
router.get_thresholds()

{'faq': 0.5, 'product': 0.5, 'smalltalk': 0.5}

In [17]:
router.set_threshold(0.40)

In [18]:
test_queries = [
    "Can I return a damaged product?",
    "Show me Campus shoes below 1200",
    "Hey, how are you?",
    "Explain quantum physics"
]

for query in test_queries:
    result = router(query)

    print("Query:", query)
    print("Route:", result.name)
    print()

Query: Can I return a damaged product?
Route: faq

Query: Show me Campus shoes below 1200
Route: product

Query: Hey, how are you?
Route: smalltalk

Query: Explain quantum physics
Route: None

